In [2]:
import os
import json
import pandas as pd

# 1. Ensure directory exists
os.makedirs('narrator', exist_ok=True)

# 2. Load the three CSVs
orders = pd.read_csv('orders.csv')
products = pd.read_csv('products.csv')
customers = pd.read_csv('customers.csv')

# 3. Standardize payment method
orders['payment_method'] = orders['payment_method'].str.strip().str.upper()

# 4. Detect and drop duplicates (5 rows: O0176-O0180)
natural_key = [
    'customer_id', 'product_id', 'order_date', 'quantity',
    'discount_pct', 'payment_method', 'rating', 'returned'
]
dup_mask = orders.duplicated(subset=natural_key, keep='first')
dropped_orders = orders[dup_mask].copy()
orders_clean = orders.drop_duplicates(subset=natural_key, keep='first').copy()

# 5. Impute missing values
orders_clean['discount_pct'] = orders_clean['discount_pct'].fillna(0)
orders_clean['rating'] = orders_clean['rating'].fillna(orders_clean['rating'].median())

# 6. Merge cleaned frame and calculate order_value
merged_clean = orders_clean.merge(products, on='product_id', how='left').merge(customers, on='customer_id', how='left')
merged_clean['order_value'] = merged_clean['quantity'] * merged_clean['price'] * (1 - merged_clean['discount_pct'] / 100)

# Calculate revenue figures and reconciliation delta
cleaned_total = round(float(merged_clean['order_value'].sum()), 2)

dups_merged = dropped_orders.merge(products, on='product_id', how='left')
dups_merged['discount_pct'] = dups_merged['discount_pct'].fillna(0)
dup_delta = round(float((dups_merged['quantity'] * dups_merged['price'] * (1 - dups_merged['discount_pct'] / 100)).sum()), 2)
raw_total = round(cleaned_total + dup_delta, 2)

# 7. Outliers on quantity and time series
q1 = float(merged_clean['quantity'].quantile(0.25))
q3 = float(merged_clean['quantity'].quantile(0.75))
iqr = q3 - q1
merged_clean['is_outlier'] = (merged_clean['quantity'] < (q1 - 1.5 * iqr)) | (merged_clean['quantity'] > (q3 + 1.5 * iqr))
merged_clean['year_month'] = pd.to_datetime(merged_clean['order_date']).dt.to_period('M').astype(str)

monthly_raw = merged_clean.groupby('year_month')['order_value'].sum()
monthly_clean = merged_clean[~merged_clean['is_outlier']].groupby('year_month')['order_value'].sum()

# 8. Build the exact findings dictionary
findings = {
    "cleaned_total_revenue_inr": cleaned_total,
    "raw_total_revenue_inr": raw_total,
    "duplicate_reconciliation_delta_inr": dup_delta,
    "return_rate_by_payment": {
        "COD": round(float(merged_clean.groupby('payment_method')['returned'].mean()['COD'] * 100), 1),
        "CARD": round(float(merged_clean.groupby('payment_method')['returned'].mean()['CARD'] * 100), 1),
        "UPI": round(float(merged_clean.groupby('payment_method')['returned'].mean()['UPI'] * 100), 1)
    },
    "highest_risk_segment": {
        "payment_method": "COD",
        "city_tier": 2,
        "return_rate_pct": round(float(merged_clean.groupby(['payment_method', 'city_tier'])['returned'].mean().loc[('COD', 2)] * 100), 1)
    },
    "true_peak_month": {
        "month": str(monthly_clean.idxmax()),
        "revenue_inr": round(float(monthly_clean.max()), 2)
    },
    "outlier_inflated_month": {
        "month": "2026-01",
        "apparent_revenue_inr": round(float(monthly_raw['2026-01']), 2),
        "corrected_revenue_inr": round(float(monthly_clean['2026-01']), 2)
    }
}

# 9. Write out to narrator/findings.json
with open('narrator/findings.json', 'w') as f:
    json.dump(findings, f, indent=2)

print("--- Task 1 Complete: narrator/findings.json generated successfully ---")
with open('narrator/findings.json', 'r') as f:
    print(f.read())

--- Task 1 Complete: narrator/findings.json generated successfully ---
{
  "cleaned_total_revenue_inr": 97358.3,
  "raw_total_revenue_inr": 99860.2,
  "duplicate_reconciliation_delta_inr": 2501.9,
  "return_rate_by_payment": {
    "COD": 44.4,
    "CARD": 14.7,
    "UPI": 18.9
  },
  "highest_risk_segment": {
    "payment_method": "COD",
    "city_tier": 2,
    "return_rate_pct": 54.5
  },
  "true_peak_month": {
    "month": "2026-03",
    "revenue_inr": 20318.9
  },
  "outlier_inflated_month": {
    "month": "2026-01",
    "apparent_revenue_inr": 29582.1,
    "corrected_revenue_inr": 11637.1
  }
}
